# Yelp Review Highlights Integration

This notebook fetches review highlights (biz_summary and biz_summary_long) for NYC restaurants using Yelp Fusion API Premium endpoints.

In [20]:
import os
import json
import requests
import time
from math import radians, sin, cos, sqrt, atan2
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
YELP_API_KEY = 'api-key'

# Yelp API configuration
YELP_SEARCH_URL = 'https://api.yelp.com/v3/businesses/search'
YELP_BUSINESS_URL = 'https://api.yelp.com/v3/businesses'
HEADERS = {'Authorization': f'Bearer {YELP_API_KEY}'}

In [21]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two coordinates in meters"""
    R = 6371000  # Earth's radius in meters
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

In [22]:
def search_yelp_business(restaurant):
    """Search for a restaurant on Yelp using name and location"""
    params = {
        'term': restaurant['name'],
        'latitude': restaurant['latitude'],
        'longitude': restaurant['longitude'],
        'limit': 5,
        'radius': 400  # 400 meters radius
    }
    
    try:
        response = requests.get(YELP_SEARCH_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()
        
        # Find best match based on distance
        best_match = None
        min_distance = float('inf')
        
        for business in data.get('businesses', []):
            if 'coordinates' in business:
                distance = haversine_distance(
                    restaurant['latitude'], 
                    restaurant['longitude'],
                    business['coordinates']['latitude'],
                    business['coordinates']['longitude']
                )
                
                # Check if within 200 meters and closer than previous matches
                if distance < 200 and distance < min_distance:
                    best_match = business
                    min_distance = distance
        
        if best_match:
            return {
                'id': best_match['id'],
                'alias': best_match['alias'],
                'name': best_match['name'],
                'distance': min_distance,
                'url': best_match.get('url'),
                'categories': best_match.get('categories', []),
                'rating': best_match.get('rating'),
                'price': best_match.get('price'),
                'review_count': best_match.get('review_count'),
                'phone': best_match.get('phone'),
                'display_phone': best_match.get('display_phone'),
                'image_url': best_match.get('image_url'),
                'transactions': best_match.get('transactions', [])
            }
        
        return None
        
    except Exception as e:
        print(f"Error searching for {restaurant['name']}: {str(e)}")
        return None

In [23]:
def get_review_highlights(business_id):
    """Fetch review highlights for a business"""
    url = f"{YELP_BUSINESS_URL}/{business_id}/review_highlights"
    
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        
        return data
        
    except Exception as e:
        print(f"  ⚠️ Error fetching highlights: {str(e)}")
        return None

In [24]:
# Load restaurant list
with open('old_restaurant_list.json', 'r', encoding='utf-8') as f:
    all_restaurants = json.load(f)

print(f"Total restaurants: {len(all_restaurants)}")

FileNotFoundError: [Errno 2] No such file or directory: 'old_restaurant_list.json'

In [ ]:
# Process restaurants from index 16 onwards (all remaining)
test_restaurants = all_restaurants
results = []
api_calls = 0
total_to_process = len(test_restaurants)

for idx, restaurant in enumerate(test_restaurants, 1):  
    print(f"\n[{idx}/{len(all_restaurants)}] Processing: {restaurant['name']}")
    print(f"  Location: {restaurant['neighborhood']}, {restaurant['borough']}")
    
    # Step 1: Search for business on Yelp
    yelp_match = search_yelp_business(restaurant)
    api_calls += 1
    
    if not yelp_match:
        print("  ❌ No match found on Yelp")
        restaurant_data = {
            **restaurant,
            'yelp_matched': False,
            'yelp_highlights': None        }
        results.append(restaurant_data)
        time.sleep(0.1)
        continue
    
    print(f"  ✓ Matched: {yelp_match['name']}")
    print(f"  Distance: {yelp_match['distance']:.1f}m")
    print(f"  Rating: {yelp_match['rating']} ({yelp_match['review_count']} reviews)")
    
    # Step 2: Fetch review highlights
    highlights = get_review_highlights(yelp_match['id'])
    api_calls += 1
    
    # Combine data
    restaurant_data = {
        **restaurant,
        'yelp_matched': True,
        'yelp_id': yelp_match['id'],
        'yelp_alias': yelp_match['alias'],
        'yelp_url': yelp_match['url'],
        'yelp_categories': yelp_match['categories'],
        'yelp_rating': yelp_match['rating'],
        'yelp_price': yelp_match['price'],
        'yelp_review_count': yelp_match['review_count'],
        'yelp_phone': yelp_match['phone'],
        'yelp_display_phone': yelp_match['display_phone'],
        'yelp_image_url': yelp_match['image_url'],
        'yelp_transactions': yelp_match['transactions'],
        'match_distance_meters': yelp_match['distance'],
        'yelp_highlights': highlights
    }
    
    results.append(restaurant_data)
    
    time.sleep(0.2)
    
    # Progress update every 50 restaurants
    if idx % 50 == 0:
        print(f"\n  === Progress: {idx - 16} / {total_to_process} completed ===")

print(f"\n\n=== Summary ===")
print(f"Total API calls used: {api_calls}")
print(f"Restaurants matched: {sum(1 for r in results if r.get('yelp_matched'))}")
print(f"Restaurants not matched: {sum(1 for r in results if not r.get('yelp_matched'))}")


[1/628] Processing: Atlantic Grill
  Location: Upper West Side, Manhattan
  ✓ Matched: Atlantic Grill
  Distance: 18.4m
  Rating: 3.7 (683 reviews)

[2/628] Processing: Serra
  Location: Flatiron District, Manhattan
  ✓ Matched: Eataly NYC Flatiron
  Distance: 15.0m
  Rating: 3.9 (6439 reviews)

[3/628] Processing: Bixi
  Location: Harlem, Manhattan
  ✓ Matched: Bixi
  Distance: 2.5m
  Rating: 4.1 (92 reviews)

[4/628] Processing: Kokomo
  Location: Williamsburg, Brooklyn
  ✓ Matched: Kokomo
  Distance: 2.2m
  Rating: 3.9 (1056 reviews)

[5/628] Processing: Gair
  Location: Dumbo, Brooklyn
  ✓ Matched: Gair
  Distance: 1.2m
  Rating: 4.6 (28 reviews)

[6/628] Processing: Felice on Hudson
  Location: West Village, Manhattan
  ✓ Matched: Felice on Hudson
  Distance: 0.0m
  Rating: 4.5 (56 reviews)

[7/628] Processing: Stout NYC Grand Central
  Location: Murray Hill, Manhattan
  ✓ Matched: Stout NYC
  Distance: 6.4m
  Rating: 3.7 (302 reviews)

[8/628] Processing: Le Jardinier
  Location

In [ ]:
# Load existing results and append new ones
output_file = 'restaurant_yelp.json'

try:
    with open(output_file, 'r') as f:
        existing_results = json.load(f)
    print(f"Loaded {len(existing_results)} existing restaurants from {output_file}")
except FileNotFoundError:
    existing_results = []
    print(f"No existing file found, will create new {output_file}")

# Append new results
existing_results.extend(results)

# Save combined results
with open(output_file, 'w') as f:
    json.dump(existing_results, f, indent=2)

print(f"Results saved to {output_file}")
print(f"File now contains {len(existing_results)} total restaurants with Yelp data")

No existing file found, will create new restaurant_yelp.json
Results saved to restaurant_yelp.json
File now contains 628 total restaurants with Yelp data


In [ ]:
# Display sample result
print("\n=== Sample Result ===")
sample = next((r for r in results if r.get('biz_summary')), results[0])
print(json.dumps({
    'name': sample['name'],
    'yelp_matched': sample.get('yelp_matched'),
    'biz_summary': sample.get('biz_summary'),
    'biz_summary_long': sample.get('biz_summary_long')
}, indent=2))


=== Sample Result ===
{
  "name": "Atlantic Grill",
  "yelp_matched": true,
  "biz_summary": null,
  "biz_summary_long": null
}


In [ ]:
not_matched_names = [r['name'] for r in results if not r.get('yelp_matched')]
print("Here are 10 restaurants that are not matched (if available):")
for name in not_matched_names[:10]:
    print("-", name)

Here are 10 restaurants that are not matched (if available):
- Palermo Argentinian Bistro - Gramercy
- Hav & Mar
- Evalyn’s Tap House
- Estiatorio Milos Hudson Yards
- Dowling’s at The Carlyle
- Margaux by La Sirène
- Akoya
- Little Fino
- Cafe Zaffri
- Café Carmellini


In [ ]:
# Transform 'yelp_categories' from list of dicts to list of titles (strings) for all existing_results

for r in existing_results:
    yc = r.get('yelp_categories')
    if isinstance(yc, list):
        # Extract 'title' from dicts only
        titles = [cat['title'] for cat in yc if isinstance(cat, dict) and 'title' in cat]
        r['yelp_categories'] = titles

# Save the updated data back to restaurant_yelp.json
with open(output_file, 'w') as f:
    json.dump(existing_results, f, indent=2)

print(f"Updated yelp_categories saved to {output_file}")


Updated yelp_categories saved to restaurant_yelp.json


In [ ]:
import copy

# Add percent and words directly to restaurant_yelp.json's data (existing_results)
def extract_highlight_word(sentence):
    # Extract the first instance of [[HIGHLIGHT]]...[[ENDHIGHLIGHT]]
    import re
    match = re.search(r'\[\[HIGHLIGHT\]\](.+?)\[\[ENDHIGHLIGHT\]\]', sentence)
    if match:
        return match.group(1)
    return None

def make_review_count_percent(review_count, total_count):
    # Avoid ZeroDivision
    if not total_count or total_count == 0:
        return "0.0%"
    percent = review_count / total_count * 100
    return f"{percent:.1f}%"

for r in existing_results:
    yh = r.get("yelp_highlights")
    total_reviews = r.get("yelp_review_count")  # Needed for percent
    if yh and isinstance(yh, dict) and "review_highlights" in yh:
        for h in yh["review_highlights"]:
            # Add or update 'words'
            sentence = h.get("sentence", "")
            word = extract_highlight_word(sentence)
            h["words"] = word if word is not None else ""
            # Add or update 'review_count_percent'
            review_count = h.get("review_count", 0)
            percent = make_review_count_percent(review_count, total_reviews)
            h["review_count_percent"] = percent

with open(output_file, "w") as f:
    json.dump(existing_results, f, indent=2)

print(f"Added 'words' and 'review_count_percent' to all records with highlights in {output_file}")


Added 'words' and 'review_count_percent' to all records with highlights in restaurant_yelp.json


In [ ]:
import json

# Load the restaurant_yelp.json file
with open("restaurant_yelp.json", "r", encoding="utf-8") as f:
    yelp_data = json.load(f)

def generate_yelp_markdown(restaurant):
    """
    Generate markdown for a restaurant in restaurant_yelp.json, similar to restaurant_guide.md
    """
    name = restaurant.get("name", "N/A")
    markdown = f"## {name}\n\n"

    # Description (from summary or description if present)
    summary = restaurant.get("summary", "")
    description = restaurant.get("description", "")
    if summary and description:
        desc = summary + " " + description
    elif summary:
        desc = summary
    elif description:
        desc = description
    else:
        desc = "N/A"
    markdown += f"**Description:** {desc}\n\n"

    # Neighborhood
    neighborhood = restaurant.get("neighborhood", "N/A")
    markdown += f"**Neighborhood:** {neighborhood}\n\n"
    
    # Website
    website = restaurant.get("website", "N/A")
    if website and website != "N/A":
        markdown += f"**Website:** [{website}]({website})\n\n"
    else:
        markdown += f"**Website:** N/A\n\n"

    # Yelp url
    yelp_url = restaurant.get("yelp_url", "N/A")
    if yelp_url and yelp_url != "N/A":
        markdown += f"**Yelp:** [{yelp_url}]({yelp_url})\n\n"
    else:
        markdown += f"**Yelp:** N/A\n\n"

    # OpenTable ID
    opentable_id = restaurant.get("opentable_id", "N/A")
    markdown += f"**OpenTable ID:** {opentable_id}\n\n"

    # Address
    address = restaurant.get("address", "N/A")
    markdown += f"**Address:** {address}\n\n"

    # Coordinates
    latitude = restaurant.get("latitude", "")
    longitude = restaurant.get("longitude", "")
    if latitude and longitude:
        markdown += f"**Coordinates:** {latitude}, {longitude}\n\n"
    else:
        markdown += f"**Coordinates:** N/A\n\n"

    # Facebook
    facebook = restaurant.get("facebook_url", "N/A")
    if facebook and facebook != "N/A":
        markdown += f"**Facebook:** [{facebook}]({facebook})\n\n"
    else:
        markdown += f"**Facebook:** N/A\n\n"

    # Instagram
    instagram = restaurant.get("instagram_url", "N/A")
    if instagram and instagram != "N/A":
        markdown += f"**Instagram:** [{instagram}]({instagram})\n\n"
    else:
        markdown += f"**Instagram:** N/A\n\n"

    # Michelin Award
    michelin = restaurant.get("michelin_award", "N/A")
    if not michelin:
        michelin = "N/A"
    markdown += f"**Michelin Award:** {michelin}\n\n"

    # NYT Top 100 Rank
    nyt_rank = restaurant.get("nyttop100_rank", "N/A")
    if not nyt_rank:
        nyt_rank = "N/A"
    markdown += f"**NYT Top 100 Rank:** {nyt_rank}\n\n"

    # Yelp rating
    rating = restaurant.get("yelp_rating")
    if rating is not None:
        markdown += f"**Yelp Rating:** {rating}\n\n"
    else:
        markdown += f"**Yelp Rating:** N/A\n\n"

    # Yelp review count
    review_count = restaurant.get("yelp_review_count")
    if review_count is not None:
        markdown += f"**Yelp Review Count:** {review_count}\n\n"
    else:
        markdown += f"**Yelp Review Count:** N/A\n\n"

    markdown += "---\n\n"
    return markdown

all_yelp_markdown = "# NYC Restaurants Yelp Guide\n\n"
for restaurant in yelp_data:
    all_yelp_markdown += generate_yelp_markdown(restaurant)

with open("restaurant_yelp_guide.md", "w", encoding="utf-8") as f:
    f.write(all_yelp_markdown)

print(f"Generated markdown for {len(yelp_data)} restaurants from restaurant_yelp.json")
print("Saved as restaurant_yelp_guide.md")


Generated markdown for 628 restaurants from restaurant_yelp.json
Saved as restaurant_yelp_guide.md


In [ ]:
# Display a sample (first restaurant)
from IPython.display import Markdown, display
display(Markdown(generate_yelp_markdown(yelp_data[0])))

## Atlantic Grill

**Description:** Just steps from Lincoln Center, this Upper West Side spot specializes in seafood of all sorts—oyster, clam and shellfish selections that change daily and market-fresh fish available simply grilled a la carte or as composed entrees.

**Neighborhood:** Upper West Side

**Website:** [https://atlanticgrill.com](https://atlanticgrill.com)

**Yelp:** [https://www.yelp.com/biz/atlantic-grill-new-york-3?adjust_creative=i_wquLiBq29RJRyMi1kIjg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=i_wquLiBq29RJRyMi1kIjg](https://www.yelp.com/biz/atlantic-grill-new-york-3?adjust_creative=i_wquLiBq29RJRyMi1kIjg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=i_wquLiBq29RJRyMi1kIjg)

**OpenTable ID:** 1175752

**Address:** 50 W. 65th St., Manhattan, NY 10023

**Coordinates:** 40.7724966, -73.9811104

**Facebook:** [https://www.facebook.com/atlanticgrillnyc](https://www.facebook.com/atlanticgrillnyc)

**Instagram:** [https://instagram.com/atlanticgrillnyc](https://instagram.com/atlanticgrillnyc)

**Michelin Award:** N/A

**NYT Top 100 Rank:** N/A

**Yelp Rating:** 3.7

**Yelp Review Count:** 683

---



In [ ]:
import json

# Load the restaurant_yelp.json file
with open("restaurant_yelp_summarization.json", "r", encoding="utf-8") as f:
    yelp_data = json.load(f)

def generate_yelp_markdown(restaurant):
    """
    Generate markdown for a restaurant in restaurant_yelp.json, similar to restaurant_guide.md
    """
    name = restaurant.get("name", "N/A")
    markdown = f"## {name}\n\n"

    # Description (from summary or description if present)
    summary = restaurant.get("summary", "")
    description = restaurant.get("description", "")
    yelp_summary = restaurant.get("yelp_summary", "")
    
    if summary and description:
        desc = summary + " " + description
    elif summary:
        desc = summary
    elif description:
        desc = description
    else:
        desc = "N/A"
    
    # Add Yelp summary if available
    if yelp_summary:
        desc += f"\n\n{yelp_summary}"
    
    markdown += f"**Description:** {desc}\n\n"

    # Neighborhood
    neighborhood = restaurant.get("neighborhood", "N/A")
    markdown += f"**Neighborhood:** {neighborhood}\n\n"
    
    # Address
    address = restaurant.get("address", "N/A")
    markdown += f"**Address:** {address}\n\n"
    
    # Coordinates
    latitude = restaurant.get("latitude", "")
    longitude = restaurant.get("longitude", "")
    if latitude and longitude:
        markdown += f"**Coordinates:** {latitude}, {longitude}\n\n"
    else:
        markdown += f"**Coordinates:** N/A\n\n"
        
    # Yelp price
    yelp_price = restaurant.get("yelp_price", "N/A")
    if yelp_price and yelp_price != "N/A":
        markdown += f"**Price:** {yelp_price}\n\n"
    else:
        markdown += f"**Price:** N/A\n\n"
        
    # Yelp transactions
    yelp_transactions = restaurant.get("yelp_transactions", "N/A")
    if yelp_transactions and yelp_transactions != "N/A":
        if isinstance(yelp_transactions, list):
            transactions_str = ", ".join(yelp_transactions)
        else:
            transactions_str = yelp_transactions
        markdown += f"**Available:** {transactions_str}\n\n"
    else:
        markdown += "**Available:** N/A\n\n"
    
    # Cuisine
    cuisine = restaurant.get("cuisine", "N/A")
    if cuisine and cuisine != "N/A":
        markdown += f"**Cuisine:** {cuisine}\n\n"
    else:
        markdown += "**Cuisine:** N/A\n\n"
    
    # Cuisine
    collection = restaurant.get("collections", "N/A")
    if collection and collection != "N/A":
        markdown += f"**Collection:** {collection}\n\n"
    else:
        markdown += "**Collection:** N/A\n\n"
        
    # Michelin Award
    michelin = restaurant.get("michelin_award", "N/A")
    if not michelin:
        michelin = "N/A"
    markdown += f"**Michelin Award:** {michelin}\n\n"

    # NYT Top 100 Rank
    nyt_rank = restaurant.get("nyttop100_rank", "N/A")
    if not nyt_rank:
        nyt_rank = "N/A"
    markdown += f"**NYT Top 100 Rank:** {nyt_rank}\n\n"
        
    # Yelp rating
    rating = restaurant.get("yelp_rating")
    if rating is not None:
        markdown += f"**Yelp Rating:** {rating}\n\n"
    else:
        markdown += f"**Yelp Rating:** N/A\n\n"
        
    
    # Yelp review count
    review_count = restaurant.get("yelp_review_count")
    if review_count is not None:
        markdown += f"**Yelp Review Count:** {review_count}\n\n"
    else:
        markdown += f"**Yelp Review Count:** N/A\n\n"
        
    
    # OpenTable ID
    opentable_id = restaurant.get("opentable_id", "N/A")
    markdown += f"**OpenTable ID:** {opentable_id}\n\n"

    # Yelp url
    yelp_url = restaurant.get("yelp_url", "N/A")
    if yelp_url and yelp_url != "N/A":
        markdown += f"**Yelp:** [{yelp_url}]({yelp_url})\n\n"
    else:
        markdown += f"**Yelp:** N/A\n\n"
    
    # Website
    website = restaurant.get("website", "N/A")
    if website and website != "N/A":
        markdown += f"**Website:** [{website}]({website})\n\n"
    else:
        markdown += f"**Website:** N/A\n\n"

    # Facebook
    facebook = restaurant.get("facebook_url", "N/A")
    if facebook and facebook != "N/A":
        markdown += f"**Facebook:** [{facebook}]({facebook})\n\n"
    else:
        markdown += f"**Facebook:** N/A\n\n"

    # Instagram
    instagram = restaurant.get("instagram_url", "N/A")
    if instagram and instagram != "N/A":
        markdown += f"**Instagram:** [{instagram}]({instagram})\n\n"
    else:
        markdown += f"**Instagram:** N/A\n\n"

    markdown += "---\n\n"
    return markdown

all_yelp_markdown = "# NYC Restaurants Yelp Guide\n\n"
for restaurant in yelp_data:
    all_yelp_markdown += generate_yelp_markdown(restaurant)

with open("restaurant_yelp_guide.md", "w", encoding="utf-8") as f:
    f.write(all_yelp_markdown)

print(f"Generated markdown for {len(yelp_data)} restaurants from restaurant_yelp.json")
print("Saved as restaurant_yelp_guide.md")

Generated markdown for 628 restaurants from restaurant_yelp.json
Saved as restaurant_yelp_guide.md
